In [0]:
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import pandas as pd


In [0]:
from pyspark.sql import functions as F

events = spark.table("day8_catalog.ecommerce.events_silver")

ml_df = (
    events
    .groupBy("user_id")
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type") == "cart", 1).otherwise(0)).alias("cart_adds"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases")
    )
)



In [0]:
pdf = ml_df.toPandas()

X = pdf[["views", "cart_adds"]]
y = pdf["purchases"]


In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [0]:
mlflow.set_experiment("/Shared/day12_mlflow_basics")


In [0]:
with mlflow.start_run(run_name="linear_regression_v1"):

    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("features", "views, cart_adds")

    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Metric
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metric("r2_score", r2)

    # Log model
    mlflow.sklearn.log_model(model, artifact_path="model")

print(f"R² Score: {r2:.4f}")


In [0]:
with mlflow.start_run(run_name="linear_regression_v2"):

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.3)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_metric("r2_score", r2)
    mlflow.sklearn.log_model(model, "model")

print(f"R² Score (v2): {r2:.4f}")
